# 05 · Frequency, attribute and complete-record baselines

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Run after the smoke test. This notebook does not open test labels.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Frequency-only baseline

In [ ]:
from oncoplate.pipeline import frequency_baseline,run_training,spec_from_cfg
from oncoplate.calibration import calibration_metrics
for head in ('independent','joint'):
    f=frequency_baseline(cfg,head)
    metrics=calibration_metrics(f['probabilities'],f['y'],f['mask'])
    write_json(p['reports']/f'frequency_{head}_validation_metrics.json',metrics)
    print(head,metrics['log_loss'])

## 2. ResNet-50 independent-attribute head

In [ ]:
spec=spec_from_cfg(cfg,'resnet50','frozen','independent',0)
print(run_training(cfg,spec))

## 3. ResNet-50 complete-record head

In [ ]:
spec=spec_from_cfg(cfg,'resnet50','frozen','joint',0)
print(run_training(cfg,spec))

## 4. Save validation metrics and per-label evidence

In [ ]:
from oncoplate.pipeline import prediction_arrays
from oncoplate.reporting import recognition_metrics
from oncoplate.calibration import probabilities
for head in ('independent','joint'):
    run=p['runs']/f'resnet50_frozen_{head}_s0';z=prediction_arrays(run/'validation_predictions.npz')
    schema=read_json(run/'target_schema.json');pr=probabilities(z['logits'])
    write_json(run/'validation_metrics.json',calibration_metrics(pr,z['y'],z['mask']))
    table=recognition_metrics(pr,z['y'],z['mask'],schema['labels'])
    write_table(run/'validation_per_label.csv',table);display(table.head())
print('Foundation milestone complete. Use 21 for foundation calibration; the flagship needs the new benchmark.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
